# 🪙 NEIRO-QUANT: BẢNG ĐIỀU KHIỂN TÁC CHIẾN ĐỊNH LƯỢNG (ALL-IN-ONE)
> **100% CME Gold Futures (GCZ26) | Tự động tính toán GEX, Volume, Open Interest & Điểm vào lệnh**
> *Chạy 1 mạch duy nhất (Ctrl + F9 hoặc bấm Run cell bên dưới) là hiển thị toàn bộ kết quả.*

In [ ]:
# @title
# ==============================================================================
# NEIRO-QUANT: BANG DIEU KHIEN TAC CHIEN DINH LUONG ALL-IN-ONE (100% CME GOLD GCZ26)
# ==============================================================================
# TU DONG 100%: Nap file CSV that -> Loc Near The Money (+/- 5 Strikes) -> Ve bieu do
# ==============================================================================

# BAT/TAT CHE DO LOC NEAR THE MONEY (Mac dinh True: Lay +/- 5 strikes quanh gia ATM cho Intraday)
NEAR_THE_MONEY_ONLY = True
MAX_NEAR_STRIKES = 10  # 10 strikes (+/- 5 strikes quanh ATM) - Chuan Intraday Gold

# NGUON DU LIEU: DAN TAY TU CLIPBOARD (Neu de trong, Colab se tu dong tim file da nap hoac mo nut Upload)
PASTE_CLIPBOARD_HERE = """

"""

import os
import re
import io
import glob
import time
import math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

t0 = time.time()

# ------------------------------------------------------------------------------
# 1. HAM CHUAN HOA SO LIEU & PARSER THONG MINH
# ------------------------------------------------------------------------------
def clean_val(v, default=0.0):
    if v is None: return default
    if isinstance(v, (int, float)): return default if (math.isnan(v) or math.isinf(v)) else float(v)
    s = str(v).strip()
    if not s or any(err in s.upper() for err in ('#ERROR', '#REF', '#VALUE', '#N/A', 'UNCH', 'NAN', 'NULL', 'NONE')) or s in ('-', '--', '.', 'N/A'): return default
    s = s.replace('"', '').replace("'", "").replace('$', '')
    while s.startswith(('=', '+')): s = s[1:].strip()
    if s.endswith(('s', 'S')) and len(s) > 1 and (s[-2].isdigit() or s[-2] == '.'): s = s[:-1].strip()
    if s.endswith('%'): s = s[:-1].strip()
    s = s.replace(',', '').replace(' ', '')
    try: return float(s)
    except Exception:
        m = re.search(r'[-+]?\d*\.?\d+', s)
        return float(m.group(0)) if m else default

def clean_strike(v):
    if v is None: return 0.0
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip().replace('"', '').replace("'", "")
    while s.startswith(('=', '+')): s = s[1:].strip()
    if s.endswith(('c', 'C', 'p', 'P')): s = s[:-1].strip()
    s = s.replace(',', '').replace(' ', '')
    try: return float(s)
    except Exception:
        m = re.search(r'\d[\d,]*\.?\d*', str(v))
        return float(m.group(0).replace(',', '')) if m else 0.0

def parse_df_from_raw(raw):
    cols_clean = [re.sub(r'[\s_\-]+', '', str(c).lower()) for c in raw.columns]
    col_map = {cols_clean[i]: raw.columns[i] for i in range(len(raw.columns))}
    stk_col = col_map.get('strike') or col_map.get('strikeprice') or raw.columns[0]
    rows = []
    if any(k in col_map for k in ['callbid', 'callvolume', 'callopeninterest', 'calllatest', 'callmid']):
        for _, r in raw.iterrows():
            stk = clean_strike(r.get(stk_col))
            if stk <= 0: continue
            rows.append({
                'strike': stk,
                'callBid': clean_val(r.get(col_map.get('callbid', ''))),
                'callAsk': clean_val(r.get(col_map.get('callask', ''))),
                'callMid': clean_val(r.get(col_map.get('callmid', ''))),
                'callLatest': clean_val(r.get(col_map.get('calllatest', ''))),
                'callVolume': clean_val(r.get(col_map.get('callvolume', col_map.get('callvol', '')))),
                'callOpenInterest': clean_val(r.get(col_map.get('callopeninterest', col_map.get('calloi', '')))),
                'callGamma': clean_val(r.get(col_map.get('callgamma', '0.0037')), default=0.0037) or 0.0037,
                'callIV': clean_val(r.get(col_map.get('calliv', '0.215')), default=0.215),
                'putBid': clean_val(r.get(col_map.get('putbid', ''))),
                'putAsk': clean_val(r.get(col_map.get('putask', ''))),
                'putMid': clean_val(r.get(col_map.get('putmid', ''))),
                'putLatest': clean_val(r.get(col_map.get('putlatest', ''))),
                'putVolume': clean_val(r.get(col_map.get('putvolume', col_map.get('putvol', '')))),
                'putOpenInterest': clean_val(r.get(col_map.get('putopeninterest', col_map.get('putoi', '')))),
                'putGamma': clean_val(r.get(col_map.get('putgamma', '0.0037')), default=0.0037) or 0.0037,
                'putIV': clean_val(r.get(col_map.get('putiv', '0.215')), default=0.215),
            })
    return pd.DataFrame(rows).sort_values('strike').reset_index(drop=True) if rows else pd.DataFrame()

# ------------------------------------------------------------------------------
# 2. NAP DU LIEU DA TANG (CLIPBOARD -> SCAN COLAB / DRIVE -> UPLOAD NUT BAM)
# ------------------------------------------------------------------------------
df_loaded = pd.DataFrame()
source_name = ""

# A. Clipboard dán tay
if PASTE_CLIPBOARD_HERE.strip():
    try:
        raw_clip = pd.read_csv(io.StringIO(PASTE_CLIPBOARD_HERE.strip()), sep='\t' if '\t' in PASTE_CLIPBOARD_HERE else ',', on_bad_lines='skip')
        df_loaded = parse_df_from_raw(raw_clip)
        if not df_loaded.empty:
            source_name = "Clipboard (Dan tay truc tiep)"
    except Exception as e:
        print(f"[CANH BAO] Loi doc clipboard: {e}")

# B. Tu dong quet thu muc Colab & Google Drive
if df_loaded.empty:
    search_dirs = ['/content', '/content/drive/MyDrive/Neiro-Quant/data', 'data', '.', '../data']
    if not os.path.exists('/content/drive/MyDrive') and os.path.exists('/content'):
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
        except Exception:
            pass

    found_files = []
    for d in search_dirs:
        if os.path.exists(d):
            found_files.extend(glob.glob(os.path.join(d, '*.csv')))
    
    prioritized_files = sorted(
        list(set(found_files)),
        key=lambda f: (
            0 if 'options_prices_gcz' in f.lower() else
            1 if 'options_prices' in f.lower() else
            2 if 'options' in f.lower() else
            3 if 'gcz' in f.lower() else 4
        )
    )

    for fpath in prioritized_files:
        try:
            raw = pd.read_csv(fpath, encoding='utf-8-sig', encoding_errors='ignore', on_bad_lines='skip')
            df_temp = parse_df_from_raw(raw)
            if not df_temp.empty and len(df_temp) >= 5:
                df_loaded = df_temp
                source_name = fpath
                break
        except Exception:
            continue

# C. Tu dong nap tu GitHub Cloud Bot (Zero-Touch Cloud Sync 100%)
if df_loaded.empty:
    github_raw_url = 'https://raw.githubusercontent.com/DaniNguyen2509/Neiro-Quant/main/data/options_prices_GCZ26.csv'
    try:
        raw_cloud = pd.read_csv(github_raw_url, on_bad_lines='skip')
        df_temp = parse_df_from_raw(raw_cloud)
        if not df_temp.empty and len(df_temp) >= 5:
            df_loaded = df_temp
            source_name = 'GitHub Cloud Sync (Tu dong 100% Zero-Touch)'
            print(f'☁️ [ZERO-TOUCH] Da nap thanh cong du lieu moi nhat tu GitHub Cloud Bot!')
    except Exception as e:
        pass

# C. Neu van chua co, mo nut Choose Files truc tiep
if df_loaded.empty and os.path.exists('/content'):
    try:
        from google.colab import files
        print("=" * 80)
        print("⚠️ [CHUA THAY FILE CSV] Hay bam nut 'Choose Files' ben duoi de tai file:")
        print("👉 Chon file 'options_prices_GCZ26.csv' tu may tinh cua anh.")
        print("=" * 80)
        uploaded = files.upload()
        for fname in uploaded.keys():
            raw = pd.read_csv(fname, encoding='utf-8-sig', encoding_errors='ignore', on_bad_lines='skip')
            df_temp = parse_df_from_raw(raw)
            if not df_temp.empty:
                df_loaded = df_temp
                source_name = f"Uploaded truc tiep ({fname})"
                break
    except Exception as e:
        print(f"[CANH BAO] Khong the mo hop thoai upload: {e}")

# D. Fallback Benchmark neu hoan toan khong co file
if df_loaded.empty:
    print("[NGUON DU LIEU] CHUA TIM THAY FILE -> Dang dung Benchmark GCZ26 du phong.")
    BENCHMARK_DATA = [
        {'strike': 4135.0, 'callBid': 133.2, 'callAsk': 133.6, 'callMid': 133.4, 'callLatest': 133.4, 'callVolume': 0.0, 'callOpenInterest': 75.0, 'callGamma': 0.00336, 'callIV': 0.21764, 'putBid': 87.2, 'putAsk': 87.6, 'putMid': 87.4, 'putLatest': 87.4, 'putVolume': 1.0, 'putOpenInterest': 37.0, 'putGamma': 0.00336, 'putIV': 0.21764},
        {'strike': 4150.0, 'callBid': 125.1, 'callAsk': 125.5, 'callMid': 125.3, 'callLatest': 125.3, 'callVolume': 17.0, 'callOpenInterest': 461.0, 'callGamma': 0.00360, 'callIV': 0.21620, 'putBid': 94.1, 'putAsk': 94.5, 'putMid': 94.3, 'putLatest': 94.3, 'putVolume': 15.0, 'putOpenInterest': 1162.0, 'putGamma': 0.00360, 'putIV': 0.21620},
        {'strike': 4175.0, 'callBid': 111.9, 'callAsk': 112.3, 'callMid': 112.1, 'callLatest': 112.1, 'callVolume': 26.0, 'callOpenInterest': 1039.0, 'callGamma': 0.00380, 'callIV': 0.21530, 'putBid': 105.9, 'putAsk': 106.3, 'putMid': 106.1, 'putLatest': 106.1, 'putVolume': 20.0, 'putOpenInterest': 437.0, 'putGamma': 0.00380, 'putIV': 0.21530},
        {'strike': 4200.0, 'callBid': 99.0, 'callAsk': 99.4, 'callMid': 99.2, 'callLatest': 99.2, 'callVolume': 646.0, 'callOpenInterest': 2444.0, 'callGamma': 0.00370, 'callIV': 0.21460, 'putBid': 118.0, 'putAsk': 118.4, 'putMid': 118.2, 'putLatest': 118.2, 'putVolume': 148.0, 'putOpenInterest': 2527.0, 'putGamma': 0.00370, 'putIV': 0.21460},
        {'strike': 4225.0, 'callBid': 86.4, 'callAsk': 86.8, 'callMid': 86.6, 'callLatest': 86.6, 'callVolume': 4.0, 'callOpenInterest': 338.0, 'callGamma': 0.00350, 'callIV': 0.21400, 'putBid': 130.4, 'putAsk': 130.8, 'putMid': 130.6, 'putLatest': 130.6, 'putVolume': 7.0, 'putOpenInterest': 264.0, 'putGamma': 0.00350, 'putIV': 0.21400}
    ]
    df_loaded = pd.DataFrame(BENCHMARK_DATA)
    source_name = "Benchmark CME GCZ26 Mau"
else:
    print(f"✅ [THANH CONG] DA NAP FILE THAT: '{source_name}' ({len(df_loaded)} strikes ban dau)")

# E. Loc Near The Money neu bat che do
if NEAR_THE_MONEY_ONLY and len(df_loaded) > MAX_NEAR_STRIKES:
    atm_idx = (df_loaded['callMid'] - df_loaded['putMid']).abs().idxmin()
    half = MAX_NEAR_STRIKES // 2
    start = max(0, atm_idx - (half - 1))
    end = min(len(df_loaded), start + MAX_NEAR_STRIKES)
    df_loaded = df_loaded.iloc[start:end].reset_index(drop=True)
    print(f"🎯 [CHE DO NEAR THE MONEY] Da loc dung {len(df_loaded)} strikes: ${df_loaded['strike'].min():.0f} -> ${df_loaded['strike'].max():.0f}")

df = df_loaded.sort_values('strike').reset_index(drop=True)

# ------------------------------------------------------------------------------
# 3. TINH TOAN MA TRAN QUANT & CAC MOC TAC CHIEN
# ------------------------------------------------------------------------------
atm_idx = (df['callMid'] - df['putMid']).abs().idxmin()
atm_strike = float(df.loc[atm_idx, 'strike'])
c_mid_atm = float(df.loc[atm_idx, 'callMid'])
p_mid_atm = float(df.loc[atm_idx, 'putMid'])
r, dte_days = 0.045, 1.0
spot_price = atm_strike + math.exp(r * (dte_days / 365.0)) * (c_mid_atm - p_mid_atm)

MULT = 100.0
scale = (spot_price ** 2) * 0.01 / 1e6
df['callGEX'] = df['callGamma'] * df['callOpenInterest'] * MULT * scale
df['putGEX']  = - (df['putGamma'] * df['putOpenInterest'] * MULT * scale)
df['netGEX']  = df['callGEX'] + df['putGEX']
df['callDeltaFlow'] = df['callVolume'] * MULT * scale
df['putDeltaFlow']  = - (df['putVolume'] * MULT * scale)
df['netDeltaFlow']  = df['callDeltaFlow'] + df['putDeltaFlow']

call_wall_row = df.loc[df['callOpenInterest'].idxmax()]
put_wall_row  = df.loc[df['putOpenInterest'].idxmax()]
call_wall = float(call_wall_row['strike'])
call_wall_oi = int(call_wall_row['callOpenInterest'])
put_wall = float(put_wall_row['strike'])
put_wall_oi = int(put_wall_row['putOpenInterest'])
total_net_gex = float(df['netGEX'].sum())
regime = 'LONG_GAMMA' if total_net_gex > 10.0 else ('SHORT_GAMMA' if total_net_gex < -10.0 else 'NEUTRAL')
squeeze_trigger = call_wall + 2.50
waterfall_trigger = put_wall - 2.50

strikes = df['strike'].values
step = float(np.diff(strikes).min()) if len(strikes) > 1 else 25.0

# Xac dinh cac cua ai tien tuyen Intraday gan Spot nhat
strikes_above = strikes[strikes > spot_price]
strike_above = float(strikes_above[0]) if len(strikes_above) > 0 else call_wall
next_above = float(strikes_above[1]) if len(strikes_above) > 1 else strike_above + step

strikes_below = strikes[strikes <= spot_price]
strike_below = float(strikes_below[-1]) if len(strikes_below) > 0 else put_wall
next_below = float(strikes_below[-2]) if len(strikes_below) > 1 else strike_below - step

buffer_up = min(step * 0.10, 2.50)
buffer_down = min(step * 0.10, 2.50)
front_run_tp = 1.00  # Quy tac an non 1 gia tranh truot khop limit

# 1. BANG TAC CHIEN INTRADAY SCALPING (Tung nac thang 20 - 25 gia)
df_intraday = pd.DataFrame([
    {'Chien Luoc': f'1. BUY MUA DUOI (Squeeze {strike_above:.0f})', 'Vung Gia Kich Hoat': f'Vuot > {strike_above + buffer_up:.2f}', 'Entry': f'{strike_above + buffer_up + 0.50:.2f}', 'Hard SL': f'{strike_above - 1.50:.2f} ($4.5)', 'Take Profit (TP)': f'{next_above - front_run_tp:.2f} (An non 1$)', 'Ghi Chu': f'MM vo tran Short -> Gamma Squeeze day gia len nac {next_above:.0f}'},
    {'Chien Luoc': f'2. SELL CAN TIEN TUYEN (Fade {strike_above:.0f})', 'Vung Gia Kich Hoat': f'{strike_above - 1.50:.2f} - {strike_above:.2f}', 'Entry': f'{strike_above - 1.00:.2f}', 'Hard SL': f'{strike_above + buffer_up + 1.50:.2f} ($5.0)', 'Take Profit (TP)': f'{spot_price:.2f} (Hoi ve Spot)', 'Ghi Chu': f'Can Vol cuc dai -> MM ban Futures chan dau kìm gia'},
    {'Chien Luoc': f'3. BUY BAT SAN ATM (Rebound {strike_below:.0f})', 'Vung Gia Kich Hoat': f'{strike_below:.2f} - {strike_below + 1.50:.2f}', 'Entry': f'{strike_below + 1.00:.2f}', 'Hard SL': f'{strike_below - 4.00:.2f} ($5.0)', 'Take Profit (TP)': f'{strike_above - front_run_tp:.2f}', 'Ghi Chu': f'Dem Put ho tro hoi phuc ky thuat trong phien'},
    {'Chien Luoc': f'4. SELL THUNG SAN ATM (Breakdown {strike_below:.0f})', 'Vung Gia Kich Hoat': f'Thung < {strike_below - buffer_down:.2f}', 'Entry': f'{strike_below - buffer_down - 0.50:.2f}', 'Hard SL': f'{strike_below + 1.50:.2f} ($4.5)', 'Take Profit (TP)': f'{next_below + front_run_tp:.2f}', 'Ghi Chu': f'Mat moc ATM Pivot -> Roi ve nac tiep theo {next_below:.0f}'}
])

# 2. BANG TAC CHIEN PHUC KICH THANH TRI (Structural Swing Walls)
df_swing = pd.DataFrame([
    {'Chien Luoc': f'5. BAN DINH CALL WALL ({call_wall:.0f})', 'Vung Gia Kich Hoat': f'{call_wall - 1.50:.2f} - {call_wall:.2f}', 'Entry': f'{call_wall - 1.00:.2f}', 'Hard SL': f'{call_wall + 4.00:.2f} ($4.0)', 'Take Profit (TP)': f'{strike_above:.2f} / {spot_price:.2f}', 'Ghi Chu': f'Call Wall {call_wall_oi:,} OI (Noc nha cuoi cung cua MM)'},
    {'Chien Luoc': f'6. MUA DAY PUT WALL ({put_wall:.0f})', 'Vung Gia Kich Hoat': f'{put_wall:.2f} - {put_wall + 1.50:.2f}', 'Entry': f'{put_wall + 0.50:.2f}', 'Hard SL': f'{put_wall - 4.00:.2f} ($4.0)', 'Take Profit (TP)': f'{spot_price:.2f} / {strike_above:.2f}', 'Ghi Chu': f'Major Put Wall {put_wall_oi:,} OI (Day be tong cuoi cung cua MM)'}
])

# ------------------------------------------------------------------------------
# 4. VE BIEU DO 2 PANEL KHONG BI DE CHU (ZERO OVERLAP - PRO QUANT LAYOUT)
# ------------------------------------------------------------------------------
strikes = df['strike'].values
net_gex = df['netGEX'].values
call_oi = df['callOpenInterest'].values
put_oi = df['putOpenInterest'].values
call_vol = df['callVolume'].values
put_vol = df['putVolume'].values

step = float(np.diff(strikes).min()) if len(strikes) > 1 else 25.0
bar_h = step * 0.55  # Thanh bar day dan (~55% slot)

plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17.5, 8.6), dpi=140, sharey=True, gridspec_kw={'width_ratios': [1.22, 1.0]})

# Khoang tho tren/duoi truc Y de chu khong bao gio bi tran vien
y_bottom = strikes.min() - step * 1.05
y_top = strikes.max() + step * 1.20
ax1.set_ylim(y_bottom, y_top)

# PANEL 1: NET GEX
colors = ['#2E7D32' if x >= 0 else '#C62828' for x in net_gex]
ax1.barh(strikes, net_gex, height=bar_h, color=colors, alpha=0.82, edgecolor='black', linewidth=0.6)
ax1.axvline(0, color='black', linewidth=1.0, linestyle='-')

max_abs_gex = max(abs(net_gex.min()), abs(net_gex.max()))
span = max(max_abs_gex * 1.55, 25.0)
ax1.set_xlim(-span, span)

# In con so Net GEX cach dau thanh bar, lech nhe o moc wall de khong de len duong ke
for s, v in zip(strikes, net_gex):
    if abs(v) > 0.2:
        offset = span * 0.025 if v >= 0 else -span * 0.025
        ha = 'left' if v >= 0 else 'right'
        ax1.text(v + offset, s, f'{v:+.1f}M', fontsize=8.5, va='center', ha=ha,
                 color='#1B5E20' if v >= 0 else '#B71C1C', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.15', facecolor='#FFFFFF', edgecolor='none', alpha=0.88),
                 zorder=6)

# Gia Spot CME va ATM Pivot (Badge hop nhat chong de chu 100%)
ax1.axhline(spot_price, color='#0D47A1', linewidth=2.0, linestyle='-', zorder=3)
ax1.axhline(atm_strike, color='#546E7A', linewidth=1.1, linestyle=':', alpha=0.8, zorder=2)
ax1.text(span * 0.42, spot_price, f'◄ SPOT CME: {spot_price:.2f} (ATM: {atm_strike:.0f})',
         color='#0D47A1', fontsize=8.8, fontweight='bold', va='center',
         bbox=dict(boxstyle='round,pad=0.28', facecolor='#E3F2FD', edgecolor='#0D47A1', linewidth=1.1, alpha=0.96),
         zorder=6)

# Cua ai tien tuyen Intraday (Badge huong dan hanh dong ngay tren bieu do)
if strike_above != call_wall:
    ax1.axhline(strike_above, color='#2E7D32', linewidth=1.2, linestyle=':', alpha=0.75, zorder=2)
    ax1.text(span * 0.42, strike_above, f'◄ CỬA ẢI {strike_above:.0f} (Squeeze > {strike_above + buffer_up:.1f} → TP {next_above - front_run_tp:.0f})',
             color='#1B5E20', fontsize=8.0, fontweight='bold', va='center',
             bbox=dict(boxstyle='round,pad=0.22', facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=0.9, alpha=0.92),
             zorder=6)

# Call Wall & Squeeze lines
ax1.axhline(call_wall, color='#6A1B9A', linewidth=1.8, linestyle='--')
ax1.axhline(squeeze_trigger, color='#00838F', linewidth=1.2, linestyle=':')

# THE CHIEN THUAT CALL WALL HOP NHAT (Dat goc tren ben trai, khong bi chong de)
call_box = (
    f"[CALL WALL {call_wall:.0f}] ({call_wall_oi} OI - KHANG CU DINH MM)\n"
    f"• Vung Ban (Fade Sell): {call_wall-1.5:.1f} - {call_wall:.1f} | SL: {call_wall+4.0:.1f} ($4.0) | TP: {spot_price:.1f}\n"
    f"• Squeeze Breakout: Vuot > {squeeze_trigger:.1f} -> Cat Short, Mua Duoi"
)
ax1.text(-span * 0.96, call_wall + step * 0.52, call_box,
         fontsize=8.0, fontweight='bold', va='center', ha='left',
         bbox=dict(boxstyle='round,pad=0.35', facecolor='#F3E5F5', edgecolor='#6A1B9A', linewidth=1.2, alpha=0.95))

# Put Wall & Waterfall lines
ax1.axhline(put_wall, color='#E65100', linewidth=1.8, linestyle='-.')
ax1.axhline(waterfall_trigger, color='#D32F2F', linewidth=1.2, linestyle=':')

# THE CHIEN THUAT PUT WALL HOP NHAT (Dat phia duoi ben phai, hoan toan tach biet khoi thanh bar do)
put_box = (
    f"[MAJOR PUT WALL {put_wall:.0f}] ({put_wall_oi} Puts - HO TRO DAY MM)\n"
    f"• Vung Mua (Rebound Buy): {put_wall:.1f} - {put_wall+1.5:.1f} | SL: {put_wall-4.0:.1f} ($4.0) | TP: {spot_price:.1f}\n"
    f"• Waterfall Breakout: Thung < {waterfall_trigger:.1f} -> Ban Thao Nhanh"
)
ax1.text(span * 0.04, put_wall - step * 0.50, put_box,
         fontsize=8.0, fontweight='bold', va='center', ha='left',
         bbox=dict(boxstyle='round,pad=0.35', facecolor='#FBE9E7', edgecolor='#E65100', linewidth=1.2, alpha=0.95))

ax1.set_title('Panel 1: Net GEX (Trieu USD) & Moc Chien Thuat Vao Lenh (SL/TP)', fontsize=11.2, fontweight='bold')
ax1.set_xlabel('Net GEX (Trieu USD / 1% Bien Dong Gia)', fontsize=9.5)
ax1.set_ylabel('CME Gold Futures Strike Price ($)', fontsize=10)
ax1.set_yticks(strikes)
ax1.set_yticklabels([f'${s:.0f}' for s in strikes], fontsize=9.5, fontweight='bold')
ax1.grid(True, linestyle=':', alpha=0.6)

# PANEL 2: VOLUME INTRADAY vs OPEN INTEREST
ax2.barh(strikes, call_oi, height=bar_h, color='#C8E6C9', alpha=0.75, label='Call OI (Nen vi the)', edgecolor='#4CAF50', linewidth=0.5)
ax2.barh(strikes, -put_oi, height=bar_h, color='#FFCDD2', alpha=0.75, label='Put OI (Nen vi the)', edgecolor='#F44336', linewidth=0.5)
ax2.barh(strikes, call_vol, height=bar_h * 0.42, color='#1B5E20', alpha=0.95, label='Call Volume Hom Nay (Luc Mua)', edgecolor='black', linewidth=0.6)
ax2.barh(strikes, -put_vol, height=bar_h * 0.42, color='#B71C1C', alpha=0.95, label='Put Volume Hom Nay (Luc Ban)', edgecolor='black', linewidth=0.6)
ax2.axvline(0, color='black', linewidth=1.0, linestyle='-')

ax2.axhline(spot_price, color='#0D47A1', linewidth=2.0, linestyle='-')
ax2.axhline(call_wall, color='#6A1B9A', linewidth=1.5, linestyle='--')
ax2.axhline(put_wall, color='#E65100', linewidth=1.5, linestyle='-.')

max_vol_oi = max(call_oi.max(), put_oi.max(), call_vol.max(), put_vol.max())
p2_span = max_vol_oi * 1.15
ax2.set_xlim(-p2_span, p2_span)

ax2.set_title('Panel 2: Dong Tien Thuc Chien (Volume Hom Nay vs Open Interest Nen)', fontsize=11.2, fontweight='bold')
ax2.set_xlabel('Put OI/Vol (-)  |  Call OI/Vol (+)', fontsize=9.5)
ax2.legend(loc='lower right', fontsize=8.2, framealpha=0.95)
ax2.grid(True, linestyle=':', alpha=0.6)

header_title = f'NEIRO-QUANT: GCZ26 GOLD OPTIONS | Gia Spot: {spot_price:.2f} | Trang thai: {regime} ({total_net_gex:.1f}M USD) | SL Cung: 4.0 USD'
fig.suptitle(header_title, fontsize=12.2, fontweight='bold', y=0.98)

plt.tight_layout()
chart_file = 'quant_truevision_map.png'
plt.savefig(chart_file, dpi=140)
plt.close()

t1 = time.time()
print(f'[NEIRO-QUANT] Tinh toan xong trong {t1 - t0:.2f}s! Spot CME: {spot_price:.2f} | Call Wall: {call_wall:.0f} ({call_wall_oi} OI) | Put Wall: {put_wall:.0f} ({put_wall_oi} OI)')
print('=' * 105)

try:
    from IPython.display import Image, display
    print('=' * 105)
    print('⚡ PHẦN 1: BẢNG TÁC CHIẾN LƯỚT SÓNG TRONG NGÀY (INTRADAY SCALPING - ĂN 20-25 GIÁ VÀNG):')
    display(df_intraday)
    print('-' * 105)
    print('🏰 PHẦN 2: BẢNG TÁC CHIẾN PHỤC KÍCH THÀNH TRÌ (STRUCTURAL SWING WALLS - ĐÁNH SÓNG LỚN):')
    display(df_swing)
    print('=' * 105)
    display(Image(chart_file, width=1200))
except Exception:
    print('PHẦN 1: INTRADAY SCALPING:')
    print(df_intraday.to_string(index=False))
    print('PHẦN 2: STRUCTURAL SWING WALLS:')
    print(df_swing.to_string(index=False))
